In [1]:
!python --version

Python 3.14.6


The system cannot find the path specified.


In [11]:
import subprocess
import shutil

from os import path
from collections import namedtuple

import parhOUwie

In [3]:
subprocess.run(["python", "--version"], capture_output=True) # stdout is bytes

CompletedProcess(args=['python', '--version'], returncode=0, stdout=b'Python 3.14.6\r\n', stderr=b'')

In [4]:
subprocess.run(["python", "--version"], capture_output=True, encoding="utf8") # stdout has been decoded as utf8

CompletedProcess(args=['python', '--version'], returncode=0, stdout='Python 3.14.6\n', stderr='')

In [5]:
subprocess.run([r"C:\Program Files\R\R-4.6.0\bin\R.exe", "--version"], capture_output=True, encoding="utf8")

CompletedProcess(args=['C:\\Program Files\\R\\R-4.6.0\\bin\\R.exe', '--version'], returncode=1, stdout='', stderr='The system cannot find the path specified.\nThe system cannot find the path specified.\nR version 4.6.0 (2026-04-24 ucrt) -- "Because it was There"\nCopyright (C) 2026 The R Foundation for Statistical Computing\nPlatform: x86_64-w64-mingw32/x64\n\nR is free software and comes with ABSOLUTELY NO WARRANTY.\nYou are welcome to redistribute it under the terms of the\nGNU General Public License versions 2 or 3.\nFor more information about these matters see\nhttps://www.gnu.org/licenses/.\n\n')

In [12]:
shutil.which(r"C:\Program Files\R\R-4.6.0\bin\R.exe")

'C:\\Program Files\\R\\R-4.6.0\\bin\\R.exe'

In [8]:
subprocess.run([r"C:\Program Files\R\R-4.6.0\bin\R.exe", "-e", "x <- 3^4;print(sqrt(x))"], capture_output=True, encoding="utf8")

CompletedProcess(args=['C:\\Program Files\\R\\R-4.6.0\\bin\\R.exe', '-e', 'x <- 3^4;print(sqrt(x))'], returncode=1, stdout='\nR version 4.6.0 (2026-04-24 ucrt) -- "Because it was There"\nCopyright (C) 2026 The R Foundation for Statistical Computing\nPlatform: x86_64-w64-mingw32/x64\n\nR is free software and comes with ABSOLUTELY NO WARRANTY.\nYou are welcome to redistribute it under certain conditions.\nType \'license()\' or \'licence()\' for distribution details.\n\n  Natural language support but running in an English locale\n\nR is a collaborative project with many contributors.\nType \'contributors()\' for more information and\n\'citation()\' on how to cite R or R packages in publications.\n\nType \'demo()\' for some demos, \'help()\' for on-line help, or\n\'help.start()\' for an HTML browser interface to help.\nType \'q()\' to quit R.\n\n> x <- 3^4;print(sqrt(x))\n[1] 9\n> \n', stderr='The system cannot find the path specified.\nThe system cannot find the path specified.\n')

In [8]:
#-------------------
# for testing
#-------------------

def model_savepath(
    model_savedir: str, continuous_trait: str, discrete_model: str, continuous_model: str, nsims: int, null_model: bool
) -> str:
    return path.join(model_savedir, f"{discrete_model}{continuous_model}_{continuous_trait}_{'CID' if null_model else 'CD'}_{nsims}.Rds")


def create_rscript(
    phylogeny: str,
    data: str,
    model_savedir: str,
    continuous_trait: str,
    discrete_model: str,
    continuous_model: str,
    nsims: int,
    null_model: bool,
) -> str:

    DISCRETE_MODELS = ("ER", "SYM", "ARD")
    CONTINUOUS_MODELS = ("OUM", "OUMA", "OUMV", "OUMVA")

    # do a few sanity checks first

    if discrete_model not in DISCRETE_MODELS:
        raise ValueError(f"Argument discrete_model must be one of {DISCRETE_MODELS}, but got {discrete_model}")

    if continuous_model not in CONTINUOUS_MODELS:
        raise ValueError(f"Argument continuous_model must be one of {CONTINUOUS_MODELS}, but got {continuous_model}")

    if not path.isfile(phylogeny):
        raise ValueError(f"{phylogeny} doesn't exist or is not a file")

    if not path.isfile(data):
        raise ValueError(f"{data} doesn't exist or is not a file")

    if not path.isdir(model_savedir):
        raise ValueError(f"{model_savedir} doesn't exist or is not a directory")

    if not model_savedir.endswith(("/")):
        raise ValueError(f"Argument model_savedir is expected to and with a '/', but {model_savedir} doesn't")

    _savepath = model_savepath(
        model_savedir=model_savedir,
        continuous_trait=continuous_trait,
        discrete_model=discrete_model,
        continuous_model=continuous_model,
        nsims=nsims,
        null_model=null_model,
    )

    return f"library('ape');library('OUwie');phylogeny <- ape::read.tree('{phylogeny}');data <- read.csv('{data}')[, c('Genus_species', 'Activity_pattern_code', '{continuous_trait}')];stopifnot(all(phylogeny$tip.label == data$Genus_species));model <- OUwie::hOUwie(phy = phylogeny, data = data, rate.cat = {2 if null_model else 1}, discrete_model = '{discrete_model}', continuous_model = '{continuous_model}', nSim = {nsims}, null.model = {'TRUE' if null_model else 'FALSE'});saveRDS(object = model, file = '{_savepath}');"


In [25]:
os.path.isfile(r"../../data/chapter2/FRED/")

False

In [6]:
os.path.isdir(r"../../data/chapter2/FRED")

True

In [27]:
os.path.isfile(r"../../data/chapter2/FREED/")

False

In [26]:
os.path.isfile(r"../../data/chapter2/FRED/subsets/final.xlsx")

True

In [13]:
create_rscript(phylogeny=r"../primateEyes.phy", data=r"../primateEyes.csv", model_savedir=r"../", continuous_trait="Skull_length", discrete_model="ER", continuous_model="OUMVA", nsims=10, null_model=True)

"library('ape');library('OUwie');phylogeny <- ape::read.tree('../primateEyes.phy');data <- read.csv('../primateEyes.csv')[, c('Genus_species', 'Activity_pattern_code', 'Skull_length')];stopifnot(all(phylogeny$tip.label == data$Genus_species));model <- OUwie::hOUwie(phy = phylogeny, data = data, rate.cat = 2, discrete_model = 'ER', continuous_model = 'OUMVA', nSim = 10, null.model = TRUE);saveRDS(object = model, file = '../EROUMVA_Skull_length_CID_10.Rds');"

In [24]:
exe = subprocess.run([r"C:\Program Files\R\R-4.6.0\bin\R.exe", "-e", create_rscript(phylogeny=r"../primateEyes.phy", data=r"../primateEyes.csv", model_savedir=r"../", continuous_trait="Skull_length", discrete_model="ER", continuous_model="OUMVA", nsims=10, null_model=True)], capture_output=True, encoding="utf8")

In [25]:
exe

CompletedProcess(args=['C:\\Program Files\\R\\R-4.6.0\\bin\\R.exe', '-e', "library('ape');library('OUwie');phylogeny <- ape::read.tree('../primateEyes.phy');data <- read.csv('../primateEyes.csv')[, c('Genus_species', 'Activity_pattern_code', 'Skull_length')];stopifnot(all(phylogeny$tip.label == data$Genus_species));model <- OUwie::hOUwie(phy = phylogeny, data = data, rate.cat = 2, discrete_model = 'ER', continuous_model = 'OUMVA', nSim = 10, null.model = TRUE);saveRDS(object = model, file = '../EROUMVA_Skull_length_CID_10.Rds');"], returncode=1, stdout='\nR version 4.6.0 (2026-04-24 ucrt) -- "Because it was There"\nCopyright (C) 2026 The R Foundation for Statistical Computing\nPlatform: x86_64-w64-mingw32/x64\n\nR is free software and comes with ABSOLUTELY NO WARRANTY.\nYou are welcome to redistribute it under certain conditions.\nType \'license()\' or \'licence()\' for distribution details.\n\n  Natural language support but running in an English locale\n\nR is a collaborative project 

In [26]:
exe.returncode

1

In [27]:
exe.stdout

'\nR version 4.6.0 (2026-04-24 ucrt) -- "Because it was There"\nCopyright (C) 2026 The R Foundation for Statistical Computing\nPlatform: x86_64-w64-mingw32/x64\n\nR is free software and comes with ABSOLUTELY NO WARRANTY.\nYou are welcome to redistribute it under certain conditions.\nType \'license()\' or \'licence()\' for distribution details.\n\n  Natural language support but running in an English locale\n\nR is a collaborative project with many contributors.\nType \'contributors()\' for more information and\n\'citation()\' on how to cite R or R packages in publications.\n\nType \'demo()\' for some demos, \'help()\' for on-line help, or\n\'help.start()\' for an HTML browser interface to help.\nType \'q()\' to quit R.\n\n> library(\'ape\');library(\'OUwie\');phylogeny <- ape::read.tree(\'../primateEyes.phy\');data <- read.csv(\'../primateEyes.csv\')[, c(\'Genus_species\', \'Activity_pattern_code\', \'Skull_length\')];stopifnot(all(phylogeny$tip.label == data$Genus_species));model <- OU

In [28]:
exe.stderr

'The system cannot find the path specified.\nThe system cannot find the path specified.\nLoading required package: corpcor\nLoading required package: nloptr\nLoading required package: geiger\nLoading required package: phytools\nLoading required package: maps\nLoading required package: RColorBrewer\nWarning: From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work.\n'

In [5]:
DISCRETE_MODELS = ("ER", "SYM", "ARD")
CONTINUOUS_MODELS = ("OUM", "OUMA", "OUMV", "OUMVA")

In [7]:
params = [parhOUwie.houwie_params(discrete=d, continuous=c, null=n) for c in CONTINUOUS_MODELS for d in DISCRETE_MODELS for n in (True, False)]

In [9]:
procs_24 = [subprocess.run([r"C:\Program Files\R\R-4.6.0\bin\R.exe", "-e", create_rscript(phylogeny=r"../primateEyes.phy", data=r"../primateEyes.csv", model_savedir=r"../", continuous_trait="Skull_length", discrete_model=param.discrete, continuous_model=param.continuous, nsims=10, null_model=param.null)], capture_output=True, encoding="utf8") for param in params]

KeyboardInterrupt: 

In [10]:
# subprocess.run() is blocking, hence won't help parallelize the runs
# we need to use subprocess.Popen

In [13]:
exe = subprocess.Popen([r"C:\Program Files\R\R-4.6.0\bin\R.exe", "-e", create_rscript(phylogeny=r"../primateEyes.phy", data=r"../primateEyes.csv", model_savedir=r"../", continuous_trait="Skull_length", discrete_model="ER", continuous_model="OUMVA", nsims=10, null_model=True)],
                       cwd=r"./", shell=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding="utf8")

In [17]:
exe.poll()

1

In [22]:
exe

<Popen: returncode: 1 args: ['C:\\Program Files\\R\\R-4.6.0\\bin\\R.exe', '-...>

In [21]:
exe.stderr.read()

'The system cannot find the path specified.\nThe system cannot find the path specified.\nLoading required package: corpcor\nLoading required package: nloptr\nLoading required package: geiger\nLoading required package: phytools\nLoading required package: maps\nLoading required package: RColorBrewer\nWarning: From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work.\n'

In [20]:
exe.stdout.read()

'\nR version 4.6.0 (2026-04-24 ucrt) -- "Because it was There"\nCopyright (C) 2026 The R Foundation for Statistical Computing\nPlatform: x86_64-w64-mingw32/x64\n\nR is free software and comes with ABSOLUTELY NO WARRANTY.\nYou are welcome to redistribute it under certain conditions.\nType \'license()\' or \'licence()\' for distribution details.\n\n  Natural language support but running in an English locale\n\nR is a collaborative project with many contributors.\nType \'contributors()\' for more information and\n\'citation()\' on how to cite R or R packages in publications.\n\nType \'demo()\' for some demos, \'help()\' for on-line help, or\n\'help.start()\' for an HTML browser interface to help.\nType \'q()\' to quit R.\n\n> library(\'ape\');library(\'OUwie\');phylogeny <- ape::read.tree(\'../primateEyes.phy\');data <- read.csv(\'../primateEyes.csv\')[, c(\'Genus_species\', \'Activity_pattern_code\', \'Skull_length\')];stopifnot(all(phylogeny$tip.label == data$Genus_species));model <- OU

In [23]:
exe.kill()

In [26]:
exe.

13264